# Spike-triggered demo

Demonstration of spike-triggered epochs and averages using `SpikeData` and xarray signals.

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DIR_PACKAGE = Path.cwd().parent
DIR_REPO = DIR_PACKAGE.parent
if str(DIR_REPO) not in sys.path:
    sys.path.insert(0, str(DIR_REPO))

from sim_data_analyzer.spike_data import SpikeData
from sim_data_analyzer.xr_diff import calc_xr_csd
from sim_data_analyzer.xr_io import load_xr
from sim_data_analyzer.xr_signal import interp_time_outliers
from sim_data_analyzer.xr_spike_triggered import calc_xr_sta


In [ ]:
# Parameters
TRIGGER_POP = 'IT3'
RATE_SIGNAL_POP = 'PV2'
T_LIMITS = (10.0, 15.0)
TIME_WIN = (-0.05, 0.05)   # seconds around each spike
LFP_Y = 600.0
CSD_Y = 600.0

FPATH_SIM_RESULT = (
    DIR_PACKAGE / 'dev_scratch' / 'data_src' / 'a1_lfp_15s' / 'data_00000_seed_1000.pkl'
)
DIRPATH_PROC = DIR_PACKAGE / 'dev_scratch' / 'data_proc' / 'a1_lfp_15s_0'
FPATH_LFP = DIRPATH_PROC / 'a1_lfp_15s_0_lfp.nc'
FPATH_RATES = DIRPATH_PROC / 'a1_lfp_15s_0_rates_dt_0.005.nc'
DIRPATH_OUT = DIRPATH_PROC / 'spike_triggered_demo'
FPATH_SPIKES = DIRPATH_OUT / f'spikes_{TRIGGER_POP.lower()}_combined_abs_s.npz'

DIRPATH_OUT.mkdir(parents=True, exist_ok=True)
FPATH_SPIKES


In [ ]:
# Load or extract combined trigger spikes in absolute seconds
if FPATH_SPIKES.exists():
    trigger_spikes = SpikeData.load(FPATH_SPIKES)
else:
    with FPATH_SIM_RESULT.open('rb') as fobj:
        sim_result = pickle.load(fobj)
    trigger_spikes = SpikeData.from_sim_result(
        sim_result,
        pop_names=[TRIGGER_POP],
        combine=True,
        t0=T_LIMITS[0],
        tmax=T_LIMITS[1],
        subtract_t0=False,
        ms=False,
        ndigits=6,
    )
    trigger_spikes.save(FPATH_SPIKES)

print(trigger_spikes.metadata)
print('Number of trigger spikes:', len(trigger_spikes.get_pop_spikes(TRIGGER_POP)[0]))
print('First trigger times:', trigger_spikes.get_pop_spikes(TRIGGER_POP)[0][:10])


In [ ]:
# Load cached LFP and firing rates
lfp = load_xr(FPATH_LFP, load=True)
lfp = interp_time_outliers(lfp).sel(time=slice(*T_LIMITS))

rates = load_xr(FPATH_RATES, load=True)
rates = rates.sel(time=slice(*T_LIMITS))

print(lfp)
print(rates)


In [ ]:
# Spike-triggered LFP for one depth channel
lfp_signal = lfp.sel(y=LFP_Y)
lfp_sta = calc_xr_sta(
    lfp_signal,
    trigger_spikes,
    TIME_WIN,
    pop_name=TRIGGER_POP,
    time_units='s',
    return_mode='both',
)

print(lfp_sta['epochs'])
print(lfp_sta['avg'])


In [ ]:
# Plot spike-triggered LFP epochs and average
X_epochs = lfp_sta['epochs']
X_avg = lfp_sta['avg']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
im = ax1.imshow(
    X_epochs.values,
    aspect='auto',
    origin='lower',
    extent=[X_epochs.time_rel.values[0], X_epochs.time_rel.values[-1], 0, X_epochs.sizes['spike']],
    cmap='viridis',
)
fig.colorbar(im, ax=ax1, label='LFP')
ax1.set_ylabel('Spike #')
ax1.set_title(f'LFP @ y={LFP_Y:g}, triggered by {TRIGGER_POP} spikes')

ax2.plot(X_avg.time_rel, X_avg.values, color='k', linewidth=2)
ax2.axvline(0, color='r', linestyle='--', linewidth=1)
ax2.set_xlabel('Time relative to spike (s)')
ax2.set_ylabel('Mean LFP')
ax2.grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
# Spike-triggered CSD for one depth channel
csd = calc_xr_csd(lfp)
csd_signal = csd.sel(y=CSD_Y)
csd_sta = calc_xr_sta(
    csd_signal,
    trigger_spikes,
    TIME_WIN,
    pop_name=TRIGGER_POP,
    time_units='s',
    return_mode='both',
)

print(csd_sta['epochs'])
print(csd_sta['avg'])


In [ ]:
# Plot spike-triggered CSD epochs and average
X_epochs = csd_sta['epochs']
X_avg = csd_sta['avg']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
im = ax1.imshow(
    X_epochs.values,
    aspect='auto',
    origin='lower',
    extent=[X_epochs.time_rel.values[0], X_epochs.time_rel.values[-1], 0, X_epochs.sizes['spike']],
    cmap='coolwarm',
)
fig.colorbar(im, ax=ax1, label='CSD')
ax1.set_ylabel('Spike #')
ax1.set_title(f'CSD @ y={CSD_Y:g}, triggered by {TRIGGER_POP} spikes')

ax2.plot(X_avg.time_rel, X_avg.values, color='k', linewidth=2)
ax2.axvline(0, color='r', linestyle='--', linewidth=1)
ax2.set_xlabel('Time relative to spike (s)')
ax2.set_ylabel('Mean CSD')
ax2.grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
# Spike-triggered firing-rate signal for one population
rate_signal = rates.sel(pop=RATE_SIGNAL_POP)
rate_sta = calc_xr_sta(
    rate_signal,
    trigger_spikes,
    TIME_WIN,
    pop_name=TRIGGER_POP,
    time_units='s',
    return_mode='both',
)

print(rate_sta['epochs'])
print(rate_sta['avg'])


In [ ]:
# Plot spike-triggered firing-rate epochs and average
X_epochs = rate_sta['epochs']
X_avg = rate_sta['avg']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
im = ax1.imshow(
    X_epochs.values,
    aspect='auto',
    origin='lower',
    extent=[X_epochs.time_rel.values[0], X_epochs.time_rel.values[-1], 0, X_epochs.sizes['spike']],
    cmap='magma',
)
fig.colorbar(im, ax=ax1, label='Rate')
ax1.set_ylabel('Spike #')
ax1.set_title(f'{RATE_SIGNAL_POP} rate, triggered by {TRIGGER_POP} spikes')

ax2.plot(X_avg.time_rel, X_avg.values, color='k', linewidth=2)
ax2.axvline(0, color='r', linestyle='--', linewidth=1)
ax2.set_xlabel('Time relative to spike (s)')
ax2.set_ylabel('Mean rate')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
